# Car Detection + Color Classification (OpenVINO)

**Flow:** Install → Config (Apply) → **Load model (API)** → One image → Sample grid → **Gradio** → **Video (optional)** → GIF preview

**Pipeline:** Detect cars → classify color per crop. Labels: **CAR - [Color]**.


## 1. Install (run once)

Kernel cwd should be this folder (`detection_classification/1`) so `%pip -r requirements.txt` works, or adjust the path. You can also run `%cd` to this directory first.


In [ ]:
%pip install -q -r requirements.txt


## 2. Imports, paths, and configuration

**Deployment root** (`BASE`): folder containing `models/`, `media/`, `output/`. **`ov_utils`** discovers it (walk up from cwd, BFS, or set **`OV_DEPLOY_BASE`** / **`GETI_DEPLOY_BASE`** to the folder that **contains** `models/`).

The code walks **up** from the kernel cwd until it finds **`ov_utils.py`** and adds that directory to **`sys.path`**.


In [ ]:
import os, sys, cv2, matplotlib.pyplot as plt

_found = False
_p = os.path.abspath(os.getcwd())
while _p:
    if os.path.isfile(os.path.join(_p, "ov_utils.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        _found = True
        break
    _parent = os.path.dirname(_p)
    if _parent == _p:
        break
    _p = _parent
if not _found:
    raise ImportError(
        "ov_utils.py not found — cd to detection_classification/1, set PYTHONPATH, or open Jupyter from the repo root."
    )

from ov_utils import (
    get_notebook_config,
    get_notebook_session,
    show_openvino_config_widgets,
    run_detect_and_classify,
    run_detect_and_classify_video,
    display_output_video,
    plot_sample_inference_grid,
)

cfg = get_notebook_config()
BASE, THRESH = cfg["BASE"], cfg["THRESH"]
IMAGE_PATH, VIDEO_PATH = cfg["IMAGE_PATH"], cfg["VIDEO_PATH"]
OUTPUT_VIDEO_PATH, OUTPUT_IMAGE_PATH = cfg["OUTPUT_VIDEO_PATH"], cfg["OUTPUT_IMAGE_PATH"]
os.makedirs(os.path.join(BASE, "output"), exist_ok=True)
print("Deployment BASE:", BASE)
show_openvino_config_widgets()


## Load model (OpenVINO API)

1. **`ov.Core()`** — Initializes OpenVINO Runtime; one Core per app is enough.
2. **`core.read_model(path)`** — Load IR (`.xml`/`.bin`) into an **`ov.Model`**.
3. **`core.compile_model(model, device_name=...)`** — Build an **`ov.CompiledModel`** for the device.
4. **Inference** — Pass inputs to the compiled model; read outputs.

Same pattern as [openvino_notebooks hello-world](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/notebooks/hello-world/hello-world.ipynb): there `device.value` is the dropdown; here **`s["device"]`** comes from **Apply**. **Apply** already compiles Detection + Classification in **`ov_utils`**; the next cell shows the three calls for the **Detection** IR only.


In [ ]:
import openvino as ov

s = get_notebook_session()
cfg = get_notebook_config(s["base_dir"])
model_xml_path = os.path.join(cfg["MODEL_DIR_DET"](s["precision"]), "model.xml")

core = ov.Core()
model = core.read_model(model=model_xml_path)
compiled_model = core.compile_model(model=model, device_name=s["device"])
print(compiled_model)


## 3. One image (latency)


In [ ]:
s = get_notebook_session()
image = cv2.imread(IMAGE_PATH)
assert image is not None, IMAGE_PATH
boxes, _, _c, result_img, lat_ms = run_detect_and_classify(
    s["det_compiled"], s["cls_compiled"], image, det_label=s["det_label"], cls_labels=s["cls_labels"],
    thresh=THRESH, return_latency=True,
)
cv2.imwrite(OUTPUT_IMAGE_PATH, result_img)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"{s['device']} | {len(boxes)} car(s) | {lat_ms:.1f} ms")
plt.show()


## 4. Sample grid (`sample_image_*` in `media/`)


In [ ]:
s = get_notebook_session()
plot_sample_inference_grid(
    s["det_compiled"], s["cls_compiled"], s["det_label"], s["cls_labels"], THRESH,
    os.path.join(BASE, "media"), s["device"],
)


## 5. Gradio app


In [ ]:
from app_gradio import create_demo, DEMO_CSS
create_demo().launch(css=DEMO_CSS)


## 6. Video inference (optional)

Requires `media/Car_video.mp4` (or your configured path). Writes `output/Car_video_out.avi`; run the next cell for a GIF preview.


In [ ]:
s = get_notebook_session()
if os.path.exists(VIDEO_PATH):
    out, frames, fps = run_detect_and_classify_video(
        s["det_compiled"], s["cls_compiled"], VIDEO_PATH, output_path=OUTPUT_VIDEO_PATH,
        thresh=THRESH, det_label=s["det_label"], cls_labels=s["cls_labels"], device=s["device"],
    )
    print(f"{frames} frames @ {fps:.1f} fps -> {out}")
else:
    print("Video not found:", VIDEO_PATH)


## 7. Display output (optional GIF preview)


In [ ]:
from IPython.display import Image, display as ipy_display
gif_path, msg = display_output_video(OUTPUT_VIDEO_PATH)
print(msg)
if gif_path:
    ipy_display(Image(filename=gif_path))
